# Lab 6 — AI Agents

This notebook is a guided local demo for the Lab 6 command-line agent. First install Ollama, run `ollama pull llama3.2:3b`, install `../requirements.txt`, and start the Ollama service. Run this notebook with the portfolio repository as the working directory, or adjust `LAB_DIR` below.

Google Calendar requires OAuth setup. RAG requires approved PDFs in `data/knowledge_base/`. The notebook does not pretend those external services have been configured. Run the Calendar exercise near the end only after setting up OAuth; event creation asks for explicit approval.

In [ ]:
from pathlib import Path
import os
import sys

LAB_DIR = Path.cwd() / 'Lab-06-AI-Agents'
if not LAB_DIR.exists():
    # Also support opening this notebook with Lab-06-AI-Agents as the working directory.
    LAB_DIR = Path.cwd()
sys.path.insert(0, str(LAB_DIR))
from agent import UniversityAgent

def confirm_google_action(action):
    print(f'\nApproval required: {action}')
    return input('Type APPROVE to continue: ').strip() == 'APPROVE'

agent = UniversityAgent(workspace=LAB_DIR, session_id='notebook-demo', confirm_google=confirm_google_action)
print('Agent initialized. Confirm Ollama is running and llama3.2:3b is installed before using model responses.')

## Core tool and memory checks

These prompts exercise the calculator, Colombo date/time, SQLite memory, Notepad text entry, and system-information tools. The user name is a demo value; replace it with a test value if desired.

In [ ]:
demo_prompts = [
    'Calculate 25 × 4',
    'What is today’s date?',
    'Open Notepad and write Hello Students',
    'My name is Alex',
    'What is my name?',
    'Show my computer specs',
]
for prompt in demo_prompts:
    print(f'\nUser: {prompt}')
    print('Agent:', agent.ask(prompt))

## Safe file and RAG exercises

By default, file creation writes into `data/generated/`. A specified destination is accepted only when it exists and is listed in `LAB6_ALLOWED_WRITE_ROOTS`; file search is read-only. For RAG, place an approved PDF in `data/knowledge_base/` and ask a question that the document answers; check that the response includes source labels. Without a PDF, the agent should say that no knowledge-base documents are available.

Notepad automation is Windows-only. Google Calendar needs OAuth credentials. Follow the setup and status notes in the [Lab README](../README.md); do not treat code presence as proof of a live integration run.

In [ ]:
print(agent.ask('Create test.txt with Hello World'))
print(agent.ask(f'Find all PDFs in {LAB_DIR}'))
print(agent.ask('According to the university documents, what is the attendance policy?'))

In [ ]:
worksheet_path = Path(r'D:\MSC AI')
if worksheet_path.is_dir():
    os.environ['LAB6_ALLOWED_SEARCH_ROOTS'] = str(worksheet_path)
    os.environ['LAB6_ALLOWED_WRITE_ROOTS'] = str(worksheet_path)
    print(agent.ask(f'Find all PDF files in {worksheet_path}'))
    print(agent.ask(f'Create test.txt in {worksheet_path} with Hello World'))
else:
    print(f'Worksheet example folder is not available: {worksheet_path}. Create the folder and rerun this cell to run the exact search/create examples.')

## Optional Google Calendar exercise (worksheet page 5)

Run this after configuring the Desktop OAuth client in `.env.example` and authorizing the account. The read checks today's calendar in Asia/Colombo. The sample event is created only if you type `APPROVE`; change its title or time before running if needed.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

print(agent.tools_by_name['calendar_today'].invoke({}))
start = (datetime.now(ZoneInfo('Asia/Colombo')) + timedelta(days=1)).replace(hour=10, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)
print(agent.tools_by_name['create_calendar_event'].invoke({
    'summary': 'Lab 6 Calendar tool test',
    'start': start.isoformat(),
    'end': end.isoformat(),
    'description': 'Created for the Lab Sheet 6 page 5 Calendar exercise.'
}))